# Dietary Data Exploration – NHANES 2017–2020

This notebook explores the NHANES dietary dataset and prepares a cleaned dietary file for the HalfFull capstone project.

Focus: nutrients that may be relevant for fatigue, such as iron, vitamin B12, vitamin D, folate, magnesium, zinc, and overall calorie/protein intake.

In [1]:
import pyreadstat
import pandas as pd
from pathlib import Path

In [2]:
raw_file = Path("../data/raw/nhanes/P_DR1TOT.xpt")
df, meta = pyreadstat.read_xport(raw_file)

print(df.shape)
df.head()

(14300, 168)


,SEQN,WTDRD1PP,WTDR2DPP,DR1DRSTZ,DR1EXMER,DRABF,DRDINT,DR1DBIH,DR1DAY,DR1LANG,...,DRD370QQ,DRD370R,DRD370RQ,DRD370S,DRD370SQ,DRD370T,DRD370TQ,DRD370U,DRD370UQ,DRD370V
0,109263.0,7619.483586,17808.067666,1.0,14.0,2.0,2.0,4.0,6.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,109264.0,8235.895818,7253.761719,1.0,81.0,2.0,2.0,5.0,6.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,109265.0,33535.080310,35612.007356,1.0,88.0,2.0,2.0,19.0,4.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,109266.0,6831.068440,5988.203624,1.0,81.0,2.0,2.0,4.0,7.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,109269.0,7875.706968,18231.925894,1.0,88.0,2.0,2.0,9.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Raw dataset

The file `P_DR1TOT.xpt` contains total nutrient intake for Day 1 of the NHANES dietary interview.

Each row represents one participant (`SEQN`), with many nutrient-related variables.
For this project, we only keep a smaller set of nutrition variables that may be relevant to fatigue.

In [3]:
columns_to_keep = [
    "SEQN",
    "DR1TKCAL",
    "DR1TPROT",
    "DR1TCARB",
    "DR1TTFAT",
    "DR1TIRON",
    "DR1TVB12",
    "DR1TVD",
    "DR1TFDFE",
    "DR1TMAGN",
    "DR1TZINC"
]

diet_df = df[columns_to_keep].copy()
diet_df.head()

,SEQN,DR1TKCAL,DR1TPROT,DR1TCARB,DR1TTFAT,DR1TIRON,DR1TVB12,DR1TVD,DR1TFDFE,DR1TMAGN,DR1TZINC
0,109263.0,1402.0,52.79,187.65,48.82,13.73,2.89,4.0,633.0,175.0,8.06
1,109264.0,1046.0,55.55,121.68,37.63,8.46,1.63,0.1,182.0,167.0,3.73
2,109265.0,1926.0,57.47,246.53,80.63,6.02,2.68,2.8,281.0,200.0,6.08
3,109266.0,1698.0,52.58,217.69,73.81,12.07,1.65,1.6,283.0,300.0,6.64
4,109269.0,1251.0,24.96,159.99,57.81,6.90,1.07,0.1,279.0,85.0,3.06


In [4]:
diet_df = diet_df.rename(columns={
    "DR1TKCAL": "calories",
    "DR1TPROT": "protein",
    "DR1TCARB": "carbs",
    "DR1TTFAT": "fat",
    "DR1TIRON": "iron",
    "DR1TVB12": "vitamin_b12",
    "DR1TVD": "vitamin_d",
    "DR1TFDFE": "folate",
    "DR1TMAGN": "magnesium",
    "DR1TZINC": "zinc"
})

diet_df.head()

,SEQN,calories,protein,carbs,fat,iron,vitamin_b12,vitamin_d,folate,magnesium,zinc
0,109263.0,1402.0,52.79,187.65,48.82,13.73,2.89,4.0,633.0,175.0,8.06
1,109264.0,1046.0,55.55,121.68,37.63,8.46,1.63,0.1,182.0,167.0,3.73
2,109265.0,1926.0,57.47,246.53,80.63,6.02,2.68,2.8,281.0,200.0,6.08
3,109266.0,1698.0,52.58,217.69,73.81,12.07,1.65,1.6,283.0,300.0,6.64
4,109269.0,1251.0,24.96,159.99,57.81,6.90,1.07,0.1,279.0,85.0,3.06


In [5]:
diet_df.isna().sum()

SEQN              0
calories       1908
protein        1908
carbs          1908
fat            1908
iron           1908
vitamin_b12    1908
vitamin_d      1908
folate         1908
magnesium      1908
zinc           1908
dtype: int64

## Missing values

Some participants have missing values in the dietary variables.
This is expected in NHANES, because not every participant completed every part of the survey.

At this stage, we keep these rows and document the missingness.
Later, during dataset merging and modeling, we can decide whether to drop or impute them.

In [6]:
print("Shape:", diet_df.shape)
print("Unique participants:", diet_df["SEQN"].nunique())

Shape: (14300, 11)
Unique participants: 14300


In [7]:
output_file = Path("../data/processed/dietary_clean.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

diet_df.to_csv(output_file, index=False)
print(f"Saved cleaned file to: {output_file}")

Saved cleaned file to: ..\data\processed\dietary_clean.csv


## Conclusion

We created a cleaned dietary dataset with one row per participant and a reduced set of nutrition variables relevant for fatigue analysis.

This processed file can later be merged with:
- questionnaire data
- laboratory data
- examination data
- demographics data

using the participant ID `SEQN`.